In [1]:
documents = {
    "doc1": "AI is transforming industries. It helps in automation.",
    "doc2": "Machine learning is a subset of AI.",
}

In [2]:
from sentence_transformers import SentenceTransformer
import hashlib

# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

d:\RAG Projects\Build-a-RAG-Chatbot\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1052.76it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# Simulated vector DB
vector_db = {}   # {chunk_id: {"embedding": ..., "text": ..., "hash": ...}}

# ---- Helper Functions ---- #

def chunk_text(text, chunk_size=50):
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

def generate_hash(text):
    return hashlib.md5(text.encode()).hexdigest()

# ---- Main Processing Function ---- #

def process_documents(documents):
    for doc_id, text in documents.items():
        chunks = chunk_text(text)

        for i, chunk in enumerate(chunks):
            chunk_id = f"{doc_id}_chunk_{i}"
            new_hash = generate_hash(chunk)

            # Check if already exists
            if chunk_id in vector_db:
                if vector_db[chunk_id]["hash"] == new_hash:
                    continue  # No change → skip

            # Generate embedding
            embedding = model.encode(chunk)

            # Upsert into vector DB
            vector_db[chunk_id] = {
                "embedding": embedding,
                "text": chunk,
                "hash": new_hash
            }

# ---- Run Initial Load ---- #

process_documents(documents)

print("Initial DB:", len(vector_db))

Initial DB: 3


In [4]:
# Update doc1 slightly
documents["doc1"] = "AI is transforming industries. It helps in automation and analytics."

process_documents(documents)

print("Updated DB:", len(vector_db))

Updated DB: 3


In [5]:
import numpy as np

def search(query, top_k=2):
    query_embedding = model.encode(query)

    results = []
    for chunk_id, data in vector_db.items():
        score = np.dot(query_embedding, data["embedding"])
        results.append((chunk_id, score, data["text"]))

    results = sorted(results, key=lambda x: x[1], reverse=True)
    return results[:top_k]

# Test search
results = search("What is AI?")
for r in results:
    print(r)

('doc1_chunk_0', np.float32(0.73711467), 'AI is transforming industries. It helps in automat')
('doc2_chunk_0', np.float32(0.649824), 'Machine learning is a subset of AI.')
